# ML Pipeline Orchestration and Backfills

**Prerequisites:** Python basics and concepts 1–16 in the MLOps track. **Estimated time:** 40 minutes.

## Objectives

- Validate task dependencies, idempotent partition execution and bounded historical backfills.
- Explain an operational threshold and its owner response.
- Test a deterministic failure boundary.


## Concept recap and assumptions

This lesson uses synthetic local data and a transparent rule. The numeric threshold is an explicit policy example that must be calibrated to workload and SLO; it is not a universal production default. No cloud credentials or external service are used.


In [ ]:
# Load the paired dependency-light implementation from the repository root.
import runpy
impl=runpy.run_path('mlops/implementations/19-pipeline-orchestration.py')


## Baseline implementation

The implementation exposes the computation and decision separately, so metrics can be inspected before action. Read its validation checks, then change one deterministic input at a time.


In [ ]:
tasks=['extract','validate','train','publish']; deps={'validate':['extract'],'train':['validate'],'publish':['train']}
order=impl['topological_order'](tasks,deps); first=impl['run_partition'](42,tasks,deps)
print(order,first,impl['backfill_partitions'](40,42,3))
assert [x[0] for x in first]==order


## Comparison and operational threshold

Compare the metric to the stated threshold and explain who responds. Do not automate model retraining from this signal alone. A real deployment should record the threshold version, observation window, sample floor and remediation owner.


In [ ]:
try: impl['topological_order'](['a','b'],{'a':['b'],'b':['a']})
except ValueError as exc: print(exc)
else: raise AssertionError('cycle was accepted')


## Failure cases and exercise

Consider a missing observation, a boundary value and a shifted input. Which should page, block a release or only open an investigation?

**Exercise:** Change one threshold and one data point. Predict the result first, then encode it as an assertion.


In [ ]:
done=impl['run_partition'](42,tasks,deps)
assert impl['run_partition'](42,tasks,deps,completed=done)==[]


## Summary and next lessons

You exercised ml pipeline orchestration and backfills with deterministic evidence and an explicit policy boundary. Next read the linked concept note and practice explaining the response under a realistic incident scenario.
